# Customer Churn Prediction

End-to-end ML pipeline predicting telecom customer churn.

**Dataset:** IBM Telco Customer Churn — 7,043 customers, 20 features.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

pd.set_option("display.max_columns", None)

## 2. Load and clean the data

`TotalCharges` has a few blank strings for brand-new customers (0 tenure) — these get coerced to numeric and imputed with the median.

In [2]:
df = pd.read_csv("../data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df.shape)
df.head()

(7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 3. Quick look at class balance

Churn is imbalanced — worth knowing before picking metrics and whether to use `class_weight`.

In [3]:
df["Churn"].value_counts(normalize=True).round(3)

Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

## 4. Feature lists and preprocessing pipeline

Numeric features get scaled; categorical features get one-hot encoded. Both are wrapped in a single `ColumnTransformer` so the exact same transformation applies at training and prediction time.

In [4]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]
TARGET = "Churn"

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

## 5. Train/test split

In [5]:
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(5634, 19) (1409, 19)


## 6. Train and compare models

Both models use `class_weight="balanced"` to account for the churn imbalance seen above.

In [6]:
def evaluate(y_true, y_pred, y_proba):
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall": round(recall_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_true, y_proba), 4),
    }

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42),
}

results = {}
fitted = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = evaluate(y_test, y_pred, y_proba)
    fitted[name] = pipe
    print(name, results[name])

logistic_regression {'accuracy': 0.7374, 'precision': 0.5034, 'recall': 0.7834, 'f1': 0.613, 'roc_auc': 0.8414}


random_forest {'accuracy': 0.7587, 'precision': 0.5315, 'recall': 0.7674, 'f1': 0.628, 'roc_auc': 0.8416}


## 7. Results table

In [7]:
results_df = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)
results_df

,accuracy,precision,recall,f1,roc_auc
random_forest,0.7587,0.5315,0.7674,0.628,0.8416
logistic_regression,0.7374,0.5034,0.7834,0.613,0.8414


## Conclusion

Random Forest wins on ROC-AUC and is saved as the best model. `class_weight="balanced"` was used on both models rather than SMOTE — see the model-comparison-bank-marketing project for a full SMOTE vs class_weight test, where SMOTE didn't clearly outperform class_weight either.